# DEAP-CNNs Timeloop Reproduction

This notebook is a small OpticalLoop example, not a paper-specific simulator. It runs the canonical `deap_cnns` Timeloop macro with explicit workload and architecture variables, then displays raw Timeloop metrics, component breakdowns, simple device-count sanity checks, and the mapper loop text.

Run it from the `timeloop` conda environment so the vendored `workspace/scripts/utils.quick_run` path can call Timeloop.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "optical_loop.py").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Importing the CLI module bootstraps the local package name `opticalloop`.
import optical_loop  # noqa: F401

from opticalloop import TimeloopBackend, TimeloopLayerRef, TimeloopMacroConfig, TimeloopRun

## Cases

The two examples use the architecture settings discussed for DEAP-CNNs:

| Case | Workload | `N_COLUMNS` | `N_ROWS` |
| --- | --- | ---: | ---: |
| `R=10, D=12` | `deap_deepbench/bench0` | 100 | 12 |
| `R=3, D=113` | `deap_deepbench/bench1` | 9 | 113 |

`N_COLUMNS` is the wavelength-column count. For these examples it is treated as `R^2`: `100` for the large-kernel setting and `9` for the small-kernel setting.

In [ ]:
CASES = [
    {
        "name": "R10_D12_bench0",
        "workload": "deap_deepbench/bench0",
        "R": 10,
        "D": 12,
        "variables": {"N_TILES": 1, "N_Conv": 1, "N_COLUMNS": 100, "N_ROWS": 12},
    },
    {
        "name": "R3_D113_bench1",
        "workload": "deap_deepbench/bench1",
        "R": 3,
        "D": 113,
        "variables": {"N_TILES": 1, "N_Conv": 1, "N_COLUMNS": 9, "N_ROWS": 113},
    },
]

COMMON_VARIABLES = {"VOLTAGE_DAC_RESOLUTION": 7}


def make_architecture(case):
    variables = {**COMMON_VARIABLES, **case["variables"]}
    return TimeloopMacroConfig(
        macro="deap_cnns",
        system="fetch_all_lpddr4",
        variables=variables,
        architecture_key=f"deap_cnns:{case['name']}",
        max_utilization=False,
    )


def make_layer(case):
    network = case["workload"].split("/", 1)[0]
    return TimeloopLayerRef(network=network, layer_path=case["workload"])


runs = [
    TimeloopRun(
        layer=make_layer(case),
        architecture=make_architecture(case),
        metadata={"case": case["name"], "R": case["R"], "D": case["D"]},
    )
    for case in CASES
]
runs

In [ ]:
backend = TimeloopBackend()
results = dict(zip([case["name"] for case in CASES], backend.run_batch(runs, n_jobs=1)))

## Raw Timeloop Metrics

In [ ]:
metric_rows = []
for case in CASES:
    result = results[case["name"]]
    metric_rows.append({
        "case": case["name"],
        "workload": case["workload"],
        "R": case["R"],
        "D": case["D"],
        **case["variables"],
        "cycles": result.cycles,
        "latency_s": result.latency_s,
        "energy_j": result.energy_j,
        "raw_power_w": result.energy_j / result.latency_s if result.latency_s else 0.0,
        "area_mm2": result.area_mm2,
        "tops_per_w": result.tops_per_w,
    })

metrics = pd.DataFrame(metric_rows)
metrics

## Device-Count Sanity Check

These rows only count physical instances implied by the macro hierarchy. They are not used as an alternate simulator.

`laser`, `dac`, and `input_mrr` are input-side paths with `N_COLUMNS` instances per convolution unit. They sit before `channel_weight_row`, so the same input-side modulation is shared across the `D` channel rows, matching the row/column reuse logic used by `proposed_mrr`. The row maps only `C`, leaving unused rows idle when `C < D`. Timeloop cannot combine that with the stronger `spatial_must_reuse_inputs` anchor because `C` is part of the Inputs dataspace.

In [ ]:
count_rows = []
for case in CASES:
    variables = case["variables"]
    nconv = variables["N_Conv"]
    columns = variables["N_COLUMNS"]
    rows = variables["N_ROWS"]
    count_rows.append({
        "case": case["name"],
        "laser": nconv * columns,
        "dac": nconv * columns,
        "input_mrr": nconv * columns,
        "weight_mrr": nconv * columns * rows,
        "TIA": nconv * rows,
        "ADC": nconv,
    })

pd.DataFrame(count_rows)

## Raw Per-Component Breakdown

In [ ]:
breakdown_rows = []
for case in CASES:
    result = results[case["name"]]
    for component, energy_j in sorted(result.energy_breakdown.items()):
        breakdown_rows.append({
            "case": case["name"],
            "component": component,
            "energy_j": energy_j,
            "power_w": energy_j / result.latency_s if result.latency_s else 0.0,
        })

pd.DataFrame(breakdown_rows)

## Timeloop Mapping Loop Text

In [ ]:
for case in CASES:
    mapping_text = results[case["name"]].mapping_text or "<no Timeloop mapping text available>"
    display(Markdown(f"### {case['name']}\n```text\n{mapping_text}\n```"))